# E3SM S2S Land & Hydrologic Weekly Skill Maps (Weeks 1 to 8)

This notebook evaluates **Subseasonal Land & Hydrologic Prediction Skill** across **Weeks 1 to 8** (Days 1–56).

Land surface states (soil moisture, snow water equivalent, terrestrial water storage) exhibit longer memory than atmospheric variables, providing crucial predictability sources at subseasonal timescales (Weeks 2–8).

### Target Fields
- `SOILWATER_10CM`: Top-layer soil moisture (liquid + ice)
- `TSA`: 2-m surface air temperature over land
- `H2OSNO`: Snow water equivalent
- `TWS`: Total terrestrial water storage
- `FSNO_EFF`: Effective fractional snow cover

### Weekly Lead Windows
- **Week 1**: Days 1–7
- **Week 2**: Days 8–14
- **Week 3**: Days 15–21
- **Week 4**: Days 22–28
- **Week 5**: Days 29–35
- **Week 6**: Days 36–42
- **Week 7**: Days 43–49
- **Week 8**: Days 50–56

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    WEEK_NAMES,
    WEEK_LABELS,
    get_weekly_window,
    compute_weekly_anomalies,
    compute_weekly_acc,
    compute_weekly_acc_significance,
    paired_acc_difference,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    load_s2s_campaign_weekly,
    load_s2s_obs_weekly,
)

print("S2S Land & Hydrologic Diagnostics Module Loaded.")

## Configuration and Control Panel

In [ ]:
# =============================================================================
# USER CONTROL PANEL — S2S LAND WEEKLY SKILL (WEEKS 1 TO 8)
# =============================================================================

FIELD = "SOILWATER_10CM"  # Options: SOILWATER_10CM, TSA, H2OSNO, TWS, FSNO_EFF
COMPONENT = "lnd"
GRID = "180x360_aave"

LEAD_WEEKS = list(range(1, 9))
INIT_YEARS = list(range(1980, 1987))
INIT_MONTHS = [5, 11]
MEMBERS = [f"EN{i:02d}" for i in range(10)]

E3SM_CASES = {
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "label": "4DEnVar Ocean Init",
    },
    "E3SM-JRA55_FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "label": "JRA55-FOSIRL Ocean Init",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "label": "Reanalysis (BruteForce)",
    },
}

FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_acc", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Target Field: {FIELD} ({COMPONENT})")
print(f"Initialization Schedule: Years {INIT_YEARS[0]}–{INIT_YEARS[-1]}, Months {INIT_MONTHS}")
print(f"Figure Output Directory: {FIGURE_OUTDIR}")

## Step 1 — Load Daily Land Hindcasts & Aggregate to Weeks 1–8

In [ ]:
%%time
model_weekly = {}

for case_key, info in E3SM_CASES.items():
    model_weekly[case_key] = {}
    prefix = info["case_prefix"]
    print(f"Loading weekly land hindcasts for {case_key}...")
    for m in INIT_MONTHS:
        try:
            da_weekly = load_s2s_campaign_weekly(
                data_root=DEFAULT_DATA_DIR,
                case_prefix=prefix,
                years=INIT_YEARS,
                init_month=m,
                members=MEMBERS,
                field=FIELD,
                component=COMPONENT,
                grid=GRID,
                weeks=LEAD_WEEKS,
                verbose=False,
            )
            model_weekly[case_key][m] = da_weekly
            print(f"  Month {m:02d}: shape (Y={da_weekly.sizes.get('Y')}, M={da_weekly.sizes.get('M')}, L={da_weekly.sizes.get('L')})")
        except Exception as exc:
            print(f"  Month {m:02d}: could not load: {exc}")

loaded_cases = [k for k, v in model_weekly.items() if len(v) > 0]
print(f"Successfully loaded {len(loaded_cases)} cases: {loaded_cases}")

## Step 2 — Reference Dataset Alignment

Align observation reference (e.g. ERA5-Land or ensemble reference) to matching verification weeks.

In [ ]:
%%time
obs_weekly = {}

# For land variables, E3SM-Reanalysis ensemble mean provides the benchmark reference
ref_case = "E3SM-Reanalysis" if "E3SM-Reanalysis" in model_weekly else loaded_cases[0]
print(f"Using {ref_case} as land verification reference.")
for m in INIT_MONTHS:
    if m in model_weekly.get(ref_case, {}):
        obs_weekly[m] = model_weekly[ref_case][m].mean("M", skipna=True)

print("Reference dataset initialized for months:", list(obs_weekly.keys()))

## Step 3 — Compute Weekly Climatology & Anomalies (Weeks 1 to 8)

In [ ]:
%%time
anom_model = {}
anom_obs = {}

for m in INIT_MONTHS:
    if m in obs_weekly:
        anom_obs[m] = compute_weekly_anomalies(obs_weekly[m], year_dim="Y")
    for case_key in loaded_cases:
        if m in model_weekly[case_key]:
            if case_key not in anom_model:
                anom_model[case_key] = {}
            anom_model[case_key][m] = compute_weekly_anomalies(
                model_weekly[case_key][m], year_dim="Y"
            )

print("Computed land anomalies across Weeks 1 to 8.")

## Step 4 — Land Weekly ACC Skill Calculation

In [ ]:
%%time
acc_by_case_month = {}
sig_by_case_month = {}

for case_key in anom_model:
    acc_by_case_month[case_key] = {}
    sig_by_case_month[case_key] = {}
    for m in INIT_MONTHS:
        if m in anom_model[case_key] and m in anom_obs:
            acc = compute_weekly_acc(
                anom_model[case_key][m],
                anom_obs[m],
                year_dim="Y",
                lead_dim="L",
                ensemble_dim="M",
            )
            n_yr = len(anom_obs[m].Y)
            sig = compute_weekly_acc_significance(acc, n_samples=n_yr, alpha=0.05)
            acc_by_case_month[case_key][m] = acc
            sig_by_case_month[case_key][m] = sig
            print(f"ACC computed for {case_key}, Init Month {m:02d}")

print("Land ACC calculations complete.")

## Step 5 — Land Skill Evolution Maps (Weeks 1 to 8)

Displays the 2×4 panel of Weekly ACC skill across Weeks 1 to 8 for land points.

In [ ]:
%%time
def plot_land_weekly_acc(acc_da, sig_da, case_name, init_month, field_name):
    fig, axes = plt.subplots(
        nrows=2, ncols=4, figsize=(20, 9),
        subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
    )
    axes = axes.flatten()
    levels = np.linspace(-1, 1, 21)
    month_name = {5: "May", 11: "November"}.get(init_month, f"Month {init_month}")

    for idx, w in enumerate(range(1, 9)):
        ax = axes[idx]
        ax.coastlines(linewidth=0.8, color="0.2")
        ax.set_global()

        if w in acc_da.L.values:
            acc_w = acc_da.sel(L=w)
            cf = ax.contourf(
                acc_w.lon, acc_w.lat, acc_w,
                levels=levels, cmap="YlGnBu", extend="both",
                transform=ccrs.PlateCarree()
            )
        w_def = get_weekly_window(w)
        ax.set_title(w_def.label, fontsize=12, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"{field_name} Weekly ACC", fontsize=12)

    fig.suptitle(
        f"{case_name} — Land {field_name} Subseasonal Weekly ACC (Weeks 1–8)\nInitialized {month_name}",
        fontsize=16, fontweight="bold", y=0.98
    )
    plt.subplots_adjust(bottom=0.12, top=0.92, hspace=0.15, wspace=0.08)

    out_path = FIGURE_OUTDIR / f"{case_name}_lnd_{field_name}_init{init_month:02d}_acc_w1_w8.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Figure saved: {out_path}")
    plt.show()

for case_key in acc_by_case_month:
    for m in acc_by_case_month[case_key]:
        plot_land_weekly_acc(
            acc_by_case_month[case_key][m],
            sig_by_case_month[case_key][m],
            case_name=case_key,
            init_month=m,
            field_name=FIELD,
        )

## Validation & Integrity Check

In [ ]:
for case_key in acc_by_case_month:
    for m in acc_by_case_month[case_key]:
        acc_da = acc_by_case_month[case_key][m]
        assert "L" in acc_da.dims, f"L dimension missing in {case_key}"
        assert len(acc_da.L) == 8, f"Expected 8 weekly leads, found {len(acc_da.L)}"

print("Validation SUCCESS: All 8 land weekly leads verified.")